# U03 · NumPy + 张量思维

**目标**：从「一个数一个数算」升级到「整块数据一次算」。

| 章节 | 主题 |
|------|------|
| §1 | 张量 tensor 是什么 |
| §2 | 形状 shape 操作 |
| §3 | 广播 broadcasting |
| §4 | 向量化 vectorization—— 抛弃循环 |
| §5 | batch 概念 |
| §6 | 迷你项目预演：线性回归（练习里做） |

> 📌 学完后你就具备了**直接跳到 PyTorch**（U4）的基础，PyTorch Tensor 的 API 和 NumPy 几乎一样。

---
## §1 张量 Tensor 是什么？

**张量 = N 维数组**。把标量/向量/矩阵的概念**统一**起来。

| 名字 | 维度 | 形状示例 | 用途 |
|------|------|---------|------|
| 标量 scalar | 0 维 | `()` | 一个数 |
| 向量 vector | 1 维 | `(5,)` | 一个样本的特征 |
| 矩阵 matrix | 2 维 | `(32, 5)` | 一个 batch 的特征 |
| 3D 张量 | 3 维 | `(32, 10, 5)` | 一个 batch 的**序列**数据（32 句，每句 10 词，每词 5 维）|
| 4D 张量 | 4 维 | `(32, 3, 224, 224)` | 一个 batch 的图片 |

**翻译模型里的常见形状**：
- `(batch, seq_len)` ← 一个 batch 的句子（id 序列）
- `(batch, seq_len, emb_dim)` ← 每个词的 embedding 向量
- `(batch, seq_len, vocab_size)` ← 每一步的词表概率分布

In [1]:
import numpy as np

# 0 维：标量
s = np.array(3.14)
print('标量:', s, 'shape =', s.shape)

# 1 维：向量
v = np.array([1, 2, 3, 4, 5])
print('向量:', v, 'shape =', v.shape)

# 2 维：矩阵
m = np.array([[1, 2, 3], [4, 5, 6]])
print('矩阵 shape =', m.shape)

# 3 维：张量
t = np.zeros((2, 3, 4))   # 2 个样本，每个 3x4
print('3D 张量 shape =', t.shape)
print('维度数 ndim =', t.ndim)
print('元素总数 size =', t.size)

标量: 3.14 shape = ()
向量: [1 2 3 4 5] shape = (5,)
矩阵 shape = (2, 3)
3D 张量 shape = (2, 3, 4)
维度数 ndim = 3
元素总数 size = 24


---
## §2 形状操作 Shape

处理张量有一半时间在**调整形状**。必会 4 个操作：reshape / 转置 / 升维 / 降维。

In [2]:
import numpy as np

x = np.arange(12)             # [0,1,2,...,11]
print('原始:', x.shape)        # (12,)

# 2.1 reshape：改变形状（元素总数不变）
x2 = x.reshape(3, 4)
print('\nreshape(3,4):\n', x2)
print('新 shape:', x2.shape)

# -1 表示「剩下的维度自动算」
x3 = x.reshape(2, -1)         # 自动算出 -1 = 6
print('\nreshape(2,-1) 的形状:', x3.shape)

# 2.2 转置 .T：交换行列（仅二维有效）
print('\n转置 x2.T 的形状:', x2.T.shape)   # (3,4) -> (4,3)

# 2.3 升维：在指定位置加一个长度为 1 的维度
v = np.array([1, 2, 3])       # shape (3,)
print('\nv.shape         =', v.shape)
print('v[None, :].shape =', v[None, :].shape)    # (1, 3) 行向量
print('v[:, None].shape =', v[:, None].shape)    # (3, 1) 列向量

# 2.4 降维：去掉长度为 1 的维度
a = np.zeros((1, 3, 1))
print('\na.shape          =', a.shape)
print('squeeze 后       =', a.squeeze().shape)

原始: (12,)

reshape(3,4):
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
新 shape: (3, 4)

reshape(2,-1) 的形状: (2, 6)

转置 x2.T 的形状: (4, 3)

v.shape         = (3,)
v[None, :].shape = (1, 3)
v[:, None].shape = (3, 1)

a.shape          = (1, 3, 1)
squeeze 后       = (3,)


---
## §3 广播 Broadcasting 🔥

**广播** = 不同形状的数组可以**自动对齐**做运算，不用你手动复制数据。

### 广播规则（必记）
两个张量做逐元素运算时，从**最后一维**开始比较形状，满足以下任一条件就能对齐：
1. 两个维度相等
2. 其中一个维度是 1（自动复制）
3. 其中一个维度不存在（视为 1）

### 直观例子
```
(3, 4)    +  (4,)     → (3, 4)    ✅ 向量加到每一行
(3, 4)    +  (3, 1)   → (3, 4)    ✅ 列向量加到每一列
(3, 4)    +  (3,)     → ❌ 失败  （尾部不能对齐）
(32,10,5) +  (5,)     → (32,10,5) ✅ 向量加到每个词向量
```

In [3]:
import numpy as np

# 例 1: 矩阵 + 标量（所有元素都 +5）
A = np.array([[1, 2, 3], [4, 5, 6]])
print('A + 5:\n', A + 5)

# 例 2: 矩阵 + 向量（向量加到每一行）—— 这就是神经网络里 y = x@W + b
b = np.array([10, 20, 30])      # (3,)
print('\nA + b:\n', A + b)       # (2,3) + (3,) → (2,3)

# 例 3: 列向量 + 行向量 → 得到矩阵（外积）
col = np.array([[1], [2], [3]])    # (3,1)
row = np.array([10, 20, 30, 40])   # (4,)
print('\n(3,1) + (4,) 形状:', (col + row).shape)  # (3,4)
print(col + row)

A + 5:
 [[ 6  7  8]
 [ 9 10 11]]

A + b:
 [[11 22 33]
 [14 25 36]]

(3,1) + (4,) 形状: (3, 4)
[[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]


In [4]:
# 失败案例：看报错信息
A = np.ones((3, 4))
b = np.ones((3,))
try:
    print(A + b)
except ValueError as e:
    print('报错:', e)

# 修复：把 b 变成列向量 (3,1) 就能加
print('修复后:', (A + b[:, None]).shape)

报错: operands could not be broadcast together with shapes (3,4) (3,) 
修复后: (3, 4)


---
## §4 向量化 Vectorization — 抛弃循环 🚀

**核心原则**：写 NumPy 代码时，**能不用循环就不用循环**。

### 为什么？
- Python 循环：每次迭代要解释执行，慢
- NumPy 操作：底层 C 实现 + SIMD 指令，**快 50-1000 倍**
- GPU 训练神经网络时差距更夸张

### 例子：计算两个向量的点积

In [1]:
import numpy as np
import time

n = 1_000_000
a = np.random.randn(n)
b = np.random.randn(n)

# 方式 1: Python 循环
t0 = time.time()
result_loop = 0.0
for i in range(n):
    result_loop += a[i] * b[i]
t1 = time.time()
print(f'循环:     {t1-t0:.4f} 秒')

# 方式 2: NumPy 向量化
t0 = time.time()
result_vec = np.dot(a, b)      # 或 (a * b).sum() 或 a @ b
t1 = time.time()
print(f'向量化:   {t1-t0:.4f} 秒')

# 结果一致？
print('一致?', np.isclose(result_loop, result_vec))

循环:     0.8823 秒
向量化:   0.0637 秒
一致? True


### 向量化常用函数速查
| 需求 | 循环写法 | 向量化写法 |
|------|---------|-----------|
| 所有元素 +1 | `for i: a[i]+=1` | `a + 1` |
| 元素逐个相乘 | `for i: c[i]=a[i]*b[i]` | `a * b` |
| 点积 | 累加 | `a @ b` |
| 求和 | 累加 | `a.sum()` |
| 平均值 | 除以 n | `a.mean()` |
| 最大值索引 | 比较 | `a.argmax()` |
| 逐元素 max(0,x) | if else | `np.maximum(0, a)` |

---
## §5 batch 概念

### 为什么要 batch？
- 单样本一次训练：慢、噪声大
- 一次训全部样本：内存爆炸
- **折中：一次 32/64/128 个样本** = **mini-batch**

### 约定：**第一维永远是 batch_size**

```
x: (batch_size, in_dim)     ← 一批输入
W: (in_dim, out_dim)        ← 共享权重
y = x @ W + b               ← 输出 (batch_size, out_dim)
一次算完 batch 里所有样本！
```

从 U4 开始到 U11，你看到的每一个张量**第一维几乎都是 batch**。养成习惯：**拿到张量先看 shape**。

In [2]:
# 一批 4 个样本的线性层前向
import numpy as np

batch_size = 4
in_dim = 3
out_dim = 2

x = np.random.randn(batch_size, in_dim)
W = np.random.randn(in_dim, out_dim)
b = np.random.randn(out_dim)

y = x @ W + b        # (4,3) @ (3,2) + (2,) → (4,2)
print('输出 shape:', y.shape)
print('b 通过广播加到了每一行:', b.shape, '→ (4,2)')

# 也能单独算第 0 个样本验证
y0_single = x[0] @ W + b
print('\n单样本 y[0]:', y0_single)
print('batch  y[0]:', y[0])
print('一致?', np.allclose(y0_single, y[0]))

输出 shape: (4, 2)
b 通过广播加到了每一行: (2,) → (4,2)

单样本 y[0]: [1.08088544 3.34190701]
batch  y[0]: [1.08088544 3.34190701]
一致? True


---
## §6 预演：线性回归的 5 步训练循环

（概念先看一眼，具体你自己写在 `exercises.ipynb` 里）

拟合 $y = 2x + 3$（加点噪声）：

```
1. 前向：y_pred = x @ w + b
2. 计算 loss：L = mean((y_pred - y_true)^2)
3. 反向（求梯度）：
     dL/dw = 2 * x.T @ (y_pred - y_true) / N
     dL/db = 2 * (y_pred - y_true).mean()
4. 更新参数：
     w -= lr * dL/dw
     b -= lr * dL/db
5. 重复 1-4 几百次 → loss 越来越小
```

这个循环是**所有神经网络训练的通用模板**。U4 用 PyTorch 重写时也是这 5 步。

---
## 🎯 本章完成

跑过上面的代码后，检查下面 3 个问题能不能回答：

1. `(2,3,4)` 的张量 + `(4,)` 的向量 → 合法吗？结果 shape？
2. 有一个 100 万元素的数组 `a`，想把所有负数变 0，用**向量化**怎么写一行？
3. 一个 batch 有 32 个样本，每个样本是 10 维向量，权重 W 把它映射成 5 维。
   - x 形状？
   - W 形状？
   - y 形状？

👉 自己想答案，去 `exercises.ipynb` 做练习。